# Snapshot Active Jobs into openoppsdb

Schedule this Kaggle notebook with a cron cadence such as `0 */6 * * *`. Each scheduled run restores the previous openopps.sqlite from the Kaggle input dataset, installs the OpenOpps CLI, records active-job observations and version snapshots into that SQLite ledger, exports full CSV and Parquet table dumps, writes metadata, and versions the Kaggle dataset. Keep the notebook private when Kaggle API credentials are attached as secrets.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from datetime import UTC, datetime

DATASET_ID = os.environ.get(
    "OPENOPPS_KAGGLE_DATASET",
    "wyattowalsh/openoppsdb",
)
PACKAGE_SPEC = os.environ.get(
    "OPENOPPS_PACKAGE_SPEC",
    "git+https://github.com/wyattowalsh/openopps.git@main",
)
OUTPUT_DIR = Path(
    os.environ.get(
        "OPENOPPS_KAGGLE_OUTPUT_DIR",
        "/kaggle/working/openoppsdb",
    )
)
DB_PATH = OUTPUT_DIR / "openopps.sqlite"
KAGGLE_INPUT_DIR = Path("/kaggle/input")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run(command: list[str], *, env: dict[str, str] | None = None) -> None:
    print("+", " ".join(command))
    subprocess.run(command, check=True, env=env)

db_candidates = sorted(KAGGLE_INPUT_DIR.glob("**/openopps.sqlite"))
if db_candidates:
    source_db = max(db_candidates, key=lambda path: path.stat().st_mtime)
    shutil.copy2(source_db, DB_PATH)
    print(f"Copied prior OpenOpps DB snapshot from {source_db} to {DB_PATH}")
else:
    print("No prior OpenOpps DB snapshot found; creating a new ledger.")

run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", PACKAGE_SPEC, "kaggle"])


In [ ]:
openopps_env = os.environ.copy()
openopps_env["OPENOPPS_DB_URL"] = f"sqlite:///{DB_PATH}"
openopps_env["OPENOPPS_CACHE_ENABLED"] = "false"

run(["openopps", "admin", "db", "init"], env=openopps_env)
run(["openopps", "sync", "--metrics-json"], env=openopps_env)


In [ ]:
import sqlite3

import polars as pl

from openopps.kaggle_metadata import KAGGLE_EXPORT_CSV_DIR, KAGGLE_EXPORT_PARQUET_DIR, KAGGLE_SQLITE_TABLES
from openopps.kaggle_metadata import build_kaggle_datapackage, build_kaggle_dataset_metadata

csv_dir = OUTPUT_DIR / KAGGLE_EXPORT_CSV_DIR
parquet_dir = OUTPUT_DIR / KAGGLE_EXPORT_PARQUET_DIR
csv_dir.mkdir(parents=True, exist_ok=True)
parquet_dir.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    for table in KAGGLE_SQLITE_TABLES:
        rows = [dict(row) for row in conn.execute(f'SELECT * FROM "{table.name}"')]
        frame = pl.DataFrame(rows or {field: [] for field in table.model.model_fields})
        frame.write_csv(csv_dir / f"{table.name}.csv")
        frame.write_parquet(parquet_dir / f"{table.name}.parquet")

(OUTPUT_DIR / "dataset-metadata.json").write_text(
    json.dumps(build_kaggle_dataset_metadata(), indent=2, sort_keys=True) + "
",
    encoding="utf-8",
)
(OUTPUT_DIR / "datapackage.json").write_text(
    json.dumps(build_kaggle_datapackage(), indent=2, sort_keys=True) + "
",
    encoding="utf-8",
)

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.name.endswith(".cache.db"):
        path.unlink()
        continue
    print(path.name, path.stat().st_size)


In [ ]:
message = f"Scheduled OpenOpps active-job snapshot {datetime.now(UTC).isoformat()}"
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
token_path = os.environ.get("KAGGLE_API_V1_TOKEN_PATH")
has_kaggle_credentials = bool(
    os.environ.get("KAGGLE_API_TOKEN")
    or (token_path and Path(token_path).exists())
    or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    or kaggle_json.exists()
)

if has_kaggle_credentials:
    run([
        "kaggle",
        "datasets",
        "version",
        "-p",
        str(OUTPUT_DIR),
        "-m",
        message,
        "-q",
        "-t",
        "-r",
        "skip",
    ])
else:
    print("Skipping dataset version upload because Kaggle API credentials are unavailable.")
